# Comparing model families

Builds the same synthetic training frame as `01_end_to_end.ipynb`, then trains several model families on it and compares their test-set error, both overall and broken out by prediction horizon.

No network, no committed data: the synthetic feed is regenerated here so this notebook runs standalone.


## 1. Build the synthetic feed + training frame

(Same fixtures as `01_end_to_end.ipynb`; repeated here so this notebook is self-contained.)

In [ ]:
import json
import os
import tempfile
import zipfile
from pathlib import Path

import pandas as pd

SOURCE = "demo"
SERVICE_DAY = "20260917"

tmp_path = Path(tempfile.mkdtemp(prefix="stoptimes-cmp-"))
os.environ["STOPTIMES_HOME"] = str(tmp_path / "home")
os.environ.pop("STOPTIMES_MODEL_REGISTRY_DIR", None)
os.environ.pop("MODEL_REGISTRY_DIR", None)

from stoptimes.core.config import reset_config
reset_config()


In [ ]:
def build_static_zip(path: Path) -> Path:
    stops = ["stop_id,stop_name,stop_lat,stop_lon"]
    for i in range(4):
        stops.append(f"s{i + 1},Stop {i + 1},0.0,{i * 0.01:.5f}")
    trips = ["route_id,service_id,trip_id,direction_id,shape_id"]
    stop_times = ["trip_id,arrival_time,departure_time,stop_id,stop_sequence,shape_dist_traveled"]
    for t in range(4):
        trip_id = f"t{t + 1}"
        trips.append(f"r1,svc,{trip_id},0,sh1")
        for i in range(4):
            clock = f"{8 + t:02d}:{i * 2:02d}:00"
            stop_times.append(f"{trip_id},{clock},{clock},s{i + 1},{i + 1},{i * 1113}")
    shapes = ["shape_id,shape_pt_lat,shape_pt_lon,shape_pt_sequence,shape_dist_traveled"]
    for i in range(4):
        shapes.append(f"sh1,0.0,{i * 0.01:.5f},{i + 1},{i * 1113}")
    with zipfile.ZipFile(path, "w") as zf:
        zf.writestr("stops.txt", "\n".join(stops) + "\n")
        zf.writestr("trips.txt", "\n".join(trips) + "\n")
        zf.writestr("stop_times.txt", "\n".join(stop_times) + "\n")
        zf.writestr("shapes.txt", "\n".join(shapes) + "\n")
        zf.writestr("routes.txt", "route_id,route_short_name,route_long_name\nr1,1,Line One\n")
    return path


def build_archive(root: Path) -> Path:
    rows = []
    for t in range(4):
        start = pd.Timestamp(f"2026-09-17T{8 + t:02d}:00:00Z")
        for step in range(25):
            rows.append(
                {
                    "source_id": SOURCE,
                    "kind": "vehicle_position",
                    "trip_id": f"t{t + 1}",
                    "route_id": "r1",
                    "vehicle_id": f"v{t + 1}",
                    "start_date": SERVICE_DAY,
                    "ts": start + pd.Timedelta(seconds=15 * step),
                    "lat": 0.0,
                    "lon": round(step * 0.00125, 6),
                    "bearing": 90.0,
                }
            )
    partition = root / "year=2026" / "month=09" / "day=17" / "route_id=r1"
    partition.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_parquet(partition / "part-0.parquet")
    return root


def build_agency_dir(tmp_path: Path) -> Path:
    directory = tmp_path / "agencies"
    directory.mkdir(exist_ok=True)
    (directory / f"{SOURCE}.yaml").write_text(
        f"name: {SOURCE}\n"
        "gtfs_static_url: unused.zip\n"
        "vehicle_positions_url: http://example.invalid/vp.json\n"
        "trip_updates_url: http://example.invalid/tu.json\n"
        "poll_interval_s: 15\n"
        "timezone: UTC\n"
        "arrival_radius_m: 60.0\n"
        "dwell_s: 0.0\n"
        "shape_threshold_m: 50.0\n",
        encoding="utf-8",
    )
    return directory


static = build_static_zip(tmp_path / "gtfs.zip")
archive = build_archive(tmp_path / "archive")
agencies = build_agency_dir(tmp_path)
registry = tmp_path / "registry"


In [ ]:
from stoptimes.core.api import add_dataset, build_arrivals_dataset
from stoptimes.train.api import build_frame

add_dataset("positions", str(archive), source=SOURCE, kind="observations")
build_arrivals_dataset(
    observations="positions", static=str(static), source=SOURCE, sources_dir=str(agencies),
    out=str(tmp_path / "arrivals.parquet"), register="arrivals",
)
frame, frame_summary = build_frame(
    observations="positions", arrivals="arrivals", static=str(static), source=SOURCE,
    sources_dir=str(agencies), out=str(tmp_path / "frame.parquet"), register="frame",
)
frame_summary


## 2. Train several families on the same frame

`historical_mean` (a baseline: predicts each stop's average delay), `polyreg_distance` and `polyreg_time` (small polynomial regressions), all through the same `experiment.yaml` shape -- only `model.family` changes.

In [ ]:
FAMILIES = ["historical_mean", "polyreg_distance", "polyreg_time"]


def experiment_yaml_for(family: str) -> Path:
    path = tmp_path / f"experiment-{family}.yaml"
    path.write_text(
        f"""name: cmp-{family}
data:
  datasets: [frame]
target: arrival_delay_s
model:
  family: {family}
  params: {{}}
features:
  - distance_to_stop
  - stops_ahead
  - hour
split:
  kind: time
  train_fraction: 0.7
  test_fraction: 0.3
seed: 7
outputs:
  registry_dir: {registry}
  tags: [cmp]
baselines: []
""",
        encoding="utf-8",
    )
    return path


from stoptimes.train.api import run_experiment_from_file

trained = {family: run_experiment_from_file(experiment_yaml_for(family)) for family in FAMILIES}
{family: meta.artifact_id for family, meta in trained.items()}


## 3. Compare overall test error

In [ ]:
import pandas as pd

overall = pd.DataFrame(
    [
        {"family": family, **{k: v for k, v in meta.metrics.items() if not k.startswith("scores_by_horizon")}}
        for family, meta in trained.items()
    ]
).set_index("family").sort_values("test_mae_seconds")
overall


## 4. Compare error by prediction horizon

Each artifact's model card lists a `scores_by_horizon` breakdown; `list_models` surfaces the headline metric, but the per-horizon numbers live in the card and the metadata directly.

In [ ]:
horizon_rows = []
for family, meta in trained.items():
    for h in meta.scores_by_horizon:
        label = f"{h.horizon_min_s:.0f}-{h.horizon_max_s:.0f}s" if h.horizon_max_s is not None else f">={h.horizon_min_s:.0f}s"
        horizon_rows.append(
            {
                "family": family,
                "horizon": label,
                "n": h.n,
                "mae_s": h.mae_s,
                "rmse_s": h.rmse_s,
            }
        )

horizon_df = pd.DataFrame(horizon_rows)
horizon_df.pivot(index="horizon", columns="family", values="mae_s")


## 5. Read one family's model card

In [ ]:
from stoptimes.core.api import model_card

print(model_card(trained["historical_mean"].artifact_id, registry=registry))


## 6. List every trained artifact via the registry

In [ ]:
from stoptimes.core.api import list_models

pd.DataFrame(list_models(registry=registry)).sort_values("metric")
